# 02 - SQL in Postgres for analytics - Solution


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

PG_URL = (
    f"postgresql+psycopg2://{os.getenv('PG_USER', 'postgres')}:{os.getenv('PG_PWD', 'postgres')}"
    f"@{os.getenv('PG_HOST', 'postgres-tgt')}:{os.getenv('PG_PORT', '5432')}/{os.getenv('PG_DB', 'postgres')}"
)
engine = create_engine(PG_URL)


In [ ]:
counts_sql = text(
    "select 'orders' as table_name, count(*) as row_cnt from orders "
    "union all "
    "select 'products' as table_name, count(*) as row_cnt from products "
    "union all "
    "select 'users' as table_name, count(*) as row_cnt from users"
)
pd.read_sql(counts_sql, engine)


In [ ]:
sales_mart_sql = text(
    "with base as ("
    " select o.dt, p.category, o.amount"
    " from orders o"
    " join products p on p.product_id = o.product_id"
    ")"
    " select dt, category, count(*) as rows_cnt, round(sum(amount)::numeric, 2) as revenue"
    " from base"
    " group by dt, category"
    " order by dt, category"
)
pd.read_sql(sales_mart_sql, engine).head(20)


In [ ]:
ranking_sql = text(
    "with product_revenue as ("
    " select p.category, p.product_id, p.name, sum(o.amount) as revenue"
    " from orders o"
    " join products p on p.product_id = o.product_id"
    " group by p.category, p.product_id, p.name"
    ")"
    " select * from ("
    "   select category, product_id, name, revenue,"
    "          rank() over (partition by category order by revenue desc) as revenue_rank"
    "   from product_revenue"
    " ) ranked"
    " where revenue_rank <= 5"
    " order by category, revenue_rank"
)
pd.read_sql(ranking_sql, engine).head(20)
